<a href="https://colab.research.google.com/github/hursoo/big_k-modern_1/blob/main/gb_041_feature_dtm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1.개요

In [1]:
# 구글 드라이브 마운트

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
from collections import Counter   # 글자 수 계산에 유용한 패키지

# 2.DTM 생성: 샘플
- 텍스트의 경우 대개 단어가 '특성'으로 활용된다.

In [3]:
# 실제 데이터프레임의 열 이름은 '문서 내용'이라고 가정합니다.

data = {
    '문서ID': ['문서 1', '문서 2', '문서 3'],
    '문서내용': [
        '저는 사과 좋아요',
        '저는 바나나 좋아요',
        '저는 바나나 좋아요 저는 바나나 좋아요'
    ]
}
df = pd.DataFrame(data)

In [4]:
df

,문서ID,문서내용
0,문서 1,저는 사과 좋아요
1,문서 2,저는 바나나 좋아요
2,문서 3,저는 바나나 좋아요 저는 바나나 좋아요


In [5]:
# 모든 문서의 내용을 하나의 문자열로 합침

all_text = " ".join(df['문서내용'].astype(str))
all_text

'저는 사과 좋아요 저는 바나나 좋아요 저는 바나나 좋아요 저는 바나나 좋아요'

In [6]:
# 단어를 분리

words = all_text.split()
words

['저는', '사과', '좋아요', '저는', '바나나', '좋아요', '저는', '바나나', '좋아요', '저는', '바나나', '좋아요']

In [7]:
# 단어 빈도 계산

word_counts = Counter(words) # 리스트 words 의 각 단어 빈도를 계산
word_counts

Counter({'저는': 4, '사과': 1, '좋아요': 4, '바나나': 3})

In [8]:
# 결과를 알기 쉽게 출력
print("# 단어 종류별 빈도")
print("--------------------")

total_words = 0
for word, count in word_counts.most_common(): # 빈도가 높은 순서대로 정렬
    print(f"{word} - {count}")
    total_words += count

print("--------------------")
print(f"       {total_words}")

# 단어 종류별 빈도
--------------------
저는 - 4
좋아요 - 4
바나나 - 3
사과 - 1
--------------------
       12


In [9]:
from sklearn.feature_extraction.text import CountVectorizer

# CountVectorizer 객체 생성
# analyzer='word'는 단어 단위로 토큰화 (기본값)
# lowercase=False는 단어를 소문자로 변환하지 않음 (한글이므로 False로 설정하는 것이 좋습니다. 기본값은 True)
# token_pattern은 기본값으로 설정하여 한글 단어를 잘 인식하도록 합니다.
vectorizer = CountVectorizer(lowercase=False) # 한국어의 경우 대소문자 구분이 중요하지 않으므로 True로 해도 무방하지만, 이미지와 일치시키기 위해 False로 설정

# '문서 내용' 열의 텍스트 데이터를 사용하여 DTM 생성
# fit_transform은 어휘를 학습하고 동시에 문서들을 변환합니다.
dtm_matrix = vectorizer.fit_transform(df['문서내용'])

# DTM을 Pandas DataFrame으로 변환
# 행 이름은 문서 ID, 열 이름은 단어 (어휘)로 설정
dtm_df = pd.DataFrame(dtm_matrix.toarray(), columns=vectorizer.get_feature_names_out(), index=df['문서ID'])

# 결과 출력 (첫 번째 이미지와 동일한 형식)
print("# DTM (Document -Term Matrix: 문서-단어 행렬)")
dtm_df = dtm_df[['저는', '좋아요', '바나나', '사과']]
dtm_df

# DTM (Document -Term Matrix: 문서-단어 행렬)


,저는,좋아요,바나나,사과
문서ID,,,,
문서 1,1,1,0,1
문서 2,1,1,1,0
문서 3,2,2,2,0


# 3.문서 유사도 산출
- 코사인 유사도 지표를 사용함

In [10]:
from sklearn.metrics.pairwise import cosine_similarity

# 2. 코사인 유사도 계산
# sklearn.metrics.pairwise.cosine_similarity는 두 벡터 또는 행렬 간의 코사인 유사도를 계산합니다.
cosine_sim = cosine_similarity(dtm_matrix, dtm_matrix)

# 3. 유사도 결과 출력
print("문서 간 코사인 유사도 (TF-IDF 기반):\n")
cosine_sim_df = pd.DataFrame(cosine_sim, index=df['문서ID'], columns=df['문서ID'])
print(cosine_sim_df)

print("\n--- 개별 문서 쌍 유사도 해석 ---")
print(f"문서1과 문서2의 유사도: {cosine_sim_df.loc['문서 1', '문서 2']:.2f}")
print(f"문서1과 문서3의 유사도: {cosine_sim_df.loc['문서 1', '문서 3']:.2f}")
print(f"문서2와 문서3의 유사도: {cosine_sim_df.loc['문서 2', '문서 3']:.2f}")

문서 간 코사인 유사도 (TF-IDF 기반):

문서ID      문서 1      문서 2      문서 3
문서ID                              
문서 1  1.000000  0.666667  0.666667
문서 2  0.666667  1.000000  1.000000
문서 3  0.666667  1.000000  1.000000

--- 개별 문서 쌍 유사도 해석 ---
문서1과 문서2의 유사도: 0.67
문서1과 문서3의 유사도: 0.67
문서2와 문서3의 유사도: 1.00


# The End of Notes